# From Scratch

Base Block

In [ ]:
import torch
import torch.nn as nn
class ConvBNReLU(nn.Module):
  def __init__(self,in_c,out_c,k,s=1,p=0):
    super().__init__()
    self.block=nn.Sequential(
        nn.Conv2d(in_channels=in_c,out_channels=out_c,kernel_size=k,stride=s,padding=p,bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True)
    )
  def forward(self,x):
    return self.block(x)

Asymmetric Convolution

In [ ]:
class AsymmetricConv(nn.Module):
  def __init__(self,in_c,mid_c,out_c):
    super().__init__()
    self.block=nn.Sequential(
        ConvBNReLU(in_c,mid_c,(1,3),p=(0,1)),
        ConvBNReLU(mid_c,out_c,(3,1),p=(1,0))
    )
  def forward(self,x):
    return self.block(x)

Mini-Inception V3 Block

In [ ]:
class InceptionV3(nn.Module):
  def __init__(self,in_c):
    super().__init__()
    # 1x1
    self.branch1=ConvBNReLU(in_c,64,1)

    # 1x1 -> Asymmetric convolution
    self.branch2=nn.Sequential(
        ConvBNReLU(in_c,48,1),
        AsymmetricConv(48,64,64)
    )
    # deeper asymmetric stack
    self.branch3=nn.Sequential(
        ConvBNReLU(in_c,64,1),
        AsymmetricConv(64,96,96),
        AsymmetricConv(96,96,96)
    )
    # pooling branch
    self.branch4=nn.Sequential(
        nn.MaxPool2d(kernel_size=3,stride=1,padding=1),
        ConvBNReLU(in_c,32,1)
    )
  def forward(self,x):
    b1=self.branch1(x)
    b2=self.branch2(x)
    b3=self.branch3(x)
    b4=self.branch4(x)
    return torch.cat([b1,b2,b3,b4],dim=1)

In [ ]:
x = torch.randn(1, 192, 28, 28)
model = InceptionV3(192)

out = model(x)
print(out.shape)

torch.Size([1, 256, 28, 28])


# Transfer Learning

Training classifier only

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
model=models.inception_v3(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 211MB/s]


In [ ]:
import numpy as np
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
train_transform=transforms.Compose([
    transforms.Resize(305),
    transforms.RandomCrop(299),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize(305),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
train_data=datasets.CIFAR10(root='./data',download=True,train=True,transform=train_transform)
test_data=datasets.CIFAR10(root='./data',download=True,train=False,transform=test_transform)

train_loader=DataLoader(train_data,batch_size=64,shuffle=True,num_workers=2,pin_memory=True)
test_loader=DataLoader(test_data,batch_size=64,shuffle=False,num_workers=2,pin_memory=True)

In [ ]:
for param in model.parameters():
  param.requires_grad=False

In [ ]:
model.fc

Linear(in_features=2048, out_features=10, bias=True)

In [ ]:
num_classes=10
model.fc=nn.Linear(2048,num_classes)

In [ ]:
model.aux_logits=False

In [ ]:
import torch.optim as optim

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.fc.parameters(),lr=1e-3)

In [ ]:
model=model.to(device)

In [ ]:
epochs=3
for epoch in range(epochs):
  model.train()
  total_loss=0
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    output=model(images)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch: {epoch+1} Loss per batch: {total_loss/len(train_loader)}")

Epoch: 1 Loss per batch: 1.0571341958192304
Epoch: 2 Loss per batch: 0.8880039210362203
Epoch: 3 Loss per batch: 0.8770153142149796


In [ ]:
model.eval()
total=0
correct=0
with torch.no_grad():
  for images,labels in test_loader:
    images,labels=images.to(device),labels.to(device)
    output=model(images)
    loss=criterion(output,labels)
    _,predicted=torch.max(output,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()
  accuracy=correct/total *100
print(f"The accuracy of the model is {accuracy}%")

The accuracy of the model is 75.41%


Partial Unfreezing

In [ ]:
model.aux_logits=True

In [ ]:
for name,param in model.named_parameters():
  if "Mixed_7" in name or "fc" in name or "aux_logits" in name :
    param.requires_grad=True
  else:
    param.requires_grad=False

In [ ]:
optimizer=optim.Adam([
    {"params":model.fc.parameters(),"lr":1e-3},
    {"params":model.AuxLogits.parameters(),"lr":1e-3},
    {"params":[p for n,p in model.named_parameters()
    if "Mixed_7" in n and p.requires_grad],"lr": 1e-4}
])

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion=nn.CrossEntropyLoss()

Loss=Main Loss+0.4 x Aux Loss

In [ ]:
EPOCHS=3
for epoch in range(EPOCHS):
  model.train()
  total_loss=0
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    output,aux_outputs=model(images)

    main_loss=criterion(output,labels)
    aux_loss=criterion(aux_outputs,labels)
    loss=main_loss+ 0.4* aux_loss

    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"The total loss per batch is: {total_loss/len(train_loader)} for epoch: {epoch+1/EPOCHS}")

The total loss per batch is: 0.8973296973901941 for epoch: 0.3333333333333333
The total loss per batch is: 0.5832354224398922 for epoch: 1.3333333333333333
The total loss per batch is: 0.4952968182732992 for epoch: 2.3333333333333335


In [ ]:
model.eval()
total=0
correct=0
with torch.no_grad():
  for images,labels in test_loader:
    images,labels=images.to(device),labels.to(device)
    output=model(images)
    loss=criterion(output,labels)
    _,predicted=torch.max(output,1)
    total+=labels.size(0)
    correct+= (predicted== labels).sum().item()
  accuracy=correct/total *100

print(f"The accuracy of the model is {accuracy}%")

The accuracy of the model is 89.9%
